# 03 - Dtypes and Numerical Precision in Signal Data

This notebook explains NumPy dtypes relevant to signal processing: integers, floats, and complex types.

In [ ]:
import numpy as np

## Common NumPy Dtypes

| Dtype | Description | Bytes |
|-------|-------------|-------|
| `int16` | Signed 16-bit integer | 2 |
| `int32` | Signed 32-bit integer | 4 |
| `float32` | 32-bit float | 4 |
| `float64` | 64-bit float (default) | 8 |
| `complex64` | 2 x float32 (real + imag) | 8 |
| `complex128` | 2 x float64 (real + imag) | 16 |

In [ ]:
# Creating arrays with explicit dtypes
a_int16 = np.array([1, 2, 3], dtype=np.int16)
a_float32 = np.array([1.0, 2.0, 3.0], dtype=np.float32)
a_complex64 = np.array([1+2j, 3+4j, 5+6j], dtype=np.complex64)

print(f"int16 array:    dtype={a_int16.dtype}, itemsize={a_int16.itemsize} bytes")
print(f"float32 array:  dtype={a_float32.dtype}, itemsize={a_float32.itemsize} bytes")
print(f"complex64 array: dtype={a_complex64.dtype}, itemsize={a_complex64.itemsize} bytes")

## Complex Number Representation

In [ ]:
iq = np.array([3+4j, 1-1j, -2+2j, 0.5-0.5j])

print(f"Complex array: {iq}")
print(f"Real parts:    {np.real(iq)}")
print(f"Imag parts:    {np.imag(iq)}")
print(f"Magnitudes:    {np.abs(iq)}")
print(f"Phases (rad):  {np.angle(iq)}")
print(f"Phases (deg):  {np.degrees(np.angle(iq))}")

In [ ]:
# Real and imaginary as attributes
z = 3 + 4j
print(f"z.real = {z.real}")
print(f"z.imag = {z.imag}")
print(f"|z| = {abs(z)}")
print(f"|z| via numpy = {np.sqrt(z.real**2 + z.imag**2)}")

## Casting and Type Conversion

In [ ]:
# Float to complex (safe)
float_arr = np.array([1.0, 2.0, 3.0], dtype=np.float64)
complex_arr = float_arr.astype(np.complex128)
print(f"Float64 -> Complex128: {complex_arr}, dtype={complex_arr.dtype}")

# Complex to float (DANGEROUS: truncates imaginary part!)
complex_input = np.array([1+2j, 3+4j, 5+6j])
truncated = complex_input.astype(np.float64)
print(f"\nComplex128 -> Float64: {truncated}")
print("WARNING: imaginary parts [2, 4, 6] are silently discarded!")

In [ ]:
# Precision pitfall: float32 vs float64
val_f32 = np.float32(1.0) / np.float32(3.0)
val_f64 = np.float64(1.0) / np.float64(3.0)
print(f"1/3 in float32: {val_f32:.20f}")
print(f"1/3 in float64: {val_f64:.20f}")
print(f"Difference:      {abs(val_f64 - val_f32):.2e}")

## Overflow Pitfalls

In [ ]:
# int8 overflow
a = np.array([100], dtype=np.int8)
b = np.array([50], dtype=np.int8)
c = a + b
print(f"100 + 50 in int8 = {c}  (overflow!)")

# int16 is safer
a16 = np.array([100], dtype=np.int16)
b16 = np.array([50], dtype=np.int16)
print(f"100 + 50 in int16 = {a16 + b16}")

## Synthetic IQ Data and Dtype Mismatches

In [ ]:
# Create synthetic complex IQ signal
np.random.seed(42)
n_samples = 1024
t = np.linspace(0, 1, n_samples)
iq_signal = (1.0 + 0.5j) * np.exp(2j * np.pi * 100 * t) + 0.1 * (np.random.randn(n_samples) + 1j * np.random.randn(n_samples))

print(f"IQ signal shape: {iq_signal.shape}")
print(f"IQ signal dtype: {iq_signal.dtype}")
print(f"Mean power (direct): {np.mean(np.abs(iq_signal)**2):.4f}")

In [ ]:
# BUG: dtype mismatch in downstream calculation
# If someone accidentally casts to float64
iq_broken = iq_signal.astype(np.float64)
print(f"After cast to float64: dtype={iq_broken.dtype}")
print(f"Shape unchanged: {iq_broken.shape}")
print(f"But real part only: {iq_broken[:5]}")
print(f"\nTrying np.abs on broken data...")
print(f"abs result: {np.abs(iq_broken[:5])}")
print("The imaginary parts are gone!")

## Exercise: Detect and Fix the Dtype Bug

The following code has a bug. Run it, identify what's wrong, and fix it.

In [ ]:
# Buggy code
np.random.seed(0)
raw_iq = np.random.randn(512) + 1j * np.random.randn(512)
raw_iq = raw_iq.astype(np.float32)  # <-- BUG HERE

# This should compute average magnitude
avg_mag = np.mean(np.abs(raw_iq))
print(f"Average magnitude: {avg_mag}")
print(f"Expected around: {np.mean(np.abs(np.random.randn(512) + 1j * np.random.randn(512))):.4f}")

<details>
<summary>Solution</summary>

The bug is casting complex to float32, which discards the imaginary part. Fix:

```python
raw_iq = raw_iq.astype(np.complex64)  # keep complex type
```
</details>

## Summary

Key points:
- `complex64` = 2 x `float32`, `complex128` = 2 x `float64`
- Use `np.real()`, `np.imag()`, `np.abs()`, `np.angle()` to extract components
- Casting complex to float silently truncates the imaginary part
- For IQ data, always use complex dtypes (`complex64` or `complex128`) throughout the pipeline
- dtype mismatches are a common source of silent bugs — always verify after loading or transforming data